In [ ]:
%cd /content
!git clone https://github.com/Matiata/maxim.git
%cd /content/maxim
!pip install -r requirements.txt
!pip install -e .
%pip install -q --upgrade "jax[cuda12]==0.10.2" "flax==0.11.2"
# %rm -rf /content/maxim

/content
Cloning into 'maxim'...
remote: Enumerating objects: 524, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 524 (delta 74), reused 81 (delta 51), pack-reused 408 (from 1)
Receiving objects: 100% (524/524), 39.23 MiB | 22.92 MiB/s, done.
Resolving deltas: 100% (297/297), done.
/content/maxim
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
   

In [ ]:
from google.colab import drive # works only for colab
drive.mount('/content/gdrive/',)

Mounted at /content/gdrive/


In [ ]:
import importlib
import collections
import json
import ml_collections
import io
import time
import os
import sys
import jax.numpy as jnp
import numpy as np
import tensorflow as tf
import functools
import jax
from jax import random
from PIL import Image
from flax.core import freeze, unfreeze
from flax.traverse_util import flatten_dict, unflatten_dict
from flax.training import train_state, checkpoints
from typing import Any, Callable, Sequence, Tuple, Dict
import optax
from absl import logging

# Runtime configuration

In [ ]:
# Constants
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1  # evita OOM y coincide con MAXIM S-2 de comparacion
NUM_EPOCHS = 30
LEARNING_RATE = 2e-4
WARMUP_EPOCHS = 3
PATCH_SIZE = 256
LOG_EVERY = 10
SAVE_EVERY = 2
# Evaluacion por epoca (monitoreo).
EVAL_LOG_EVERY = 100      # imprime PSNR/loss corrientes cada N batches de validacion
MAX_EVAL_BATCHES = None  # corrida real: validacion completa de las cinco tareas
SEED = 42
NUM_EXPERTS = 5
ROUTING_MODE = "oracle"  # task_id selecciona exactamente un experto
ROUTER_ENTROPY_WEIGHT = 0.0  # desactivado durante routing oraculo
ROUTER_BALANCE_WEIGHT = 0.0  # desactivado durante routing oraculo
AUXILIARY_LOSS_WEIGHT = 1.0  # promedio ponderado de las cinco salidas auxiliares
STEPS_PER_EPOCH_OVERRIDE = 2000  # mismo presupuesto que MAXIM S-2 de comparacion
WEIGHT_DECAY = 1e-4
EVAL_PATCH_SIZE = PATCH_SIZE  # center crop determinista y OOM-safe
MODEL_VARIANT = "S-2"
OUTPUT_DIR = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/moe_oracle_all_S-2_scratch/v1_simple_expert_heads"
TRAIN_LOG_PATH = os.path.join(OUTPUT_DIR, "training.log")
ALLOW_OVERWRITE_EXISTING_RUN = True
RUN_PREFLIGHT = True  # valida el forward real antes de iniciar 60.000 pasos
DATASET_DIR = "/content/maxim/dataset"
CKPT_PATH = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/ckpt_Enhancement_LOL.npz"
_MODEL_CONFIGS = {
    "variant": "",
    "dropout_rate": 0.1,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
SAMPLING_MODE = "uniform"
TASK_DIRS = [
    "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur",
    "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze",
    "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise",
    "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain",
    "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance",
]
TASK_NAMES = tuple(os.path.basename(os.path.normpath(path)) for path in TASK_DIRS)
NUM_TASKS = len(TASK_NAMES)
assert ROUTING_MODE in ("oracle", "learned")
assert TASK_NAMES == ("deblur", "dehaze", "denoise", "derain", "enhance")
if ROUTING_MODE == "oracle":
    assert ROUTER_ENTROPY_WEIGHT == 0.0 and ROUTER_BALANCE_WEIGHT == 0.0
assert MODEL_VARIANT == "S-2", "La comparación actual usa MAXIM S-2."
assert NUM_TASKS == NUM_EXPERTS, (
    "Las metricas tarea x experto asumen una tarea por experto."
)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(
    f"REAL RUN | {MODEL_VARIANT=} | epochs={NUM_EPOCHS} | "
    f"steps/epoch={STEPS_PER_EPOCH_OVERRIDE} | eval=full deterministic"
)

REAL RUN | MODEL_VARIANT='S-2' | epochs=30 | steps/epoch=2000 | eval=full deterministic


# Trainer helpers

In [ ]:
class TrainState(train_state.TrainState):
    """Extended train state with batch statistics."""

    batch_stats: Any = None


def resize_to_match(image, target):
    """Resize image to match target dimensions using padding."""
    h, w = image.shape[0], image.shape[1]
    th, tw = target.shape[0], target.shape[1]

    # Check for transposed dimensions
    if h == tw and w == th:
        image = np.rot90(image)

    # Pad if smaller
    pad_h = max(0, th - h)
    pad_w = max(0, tw - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

    # Crop if bigger
    if h > th or w > tw:
        # Center crop
        top = (h - th) // 2
        left = (w - tw) // 2
        image = image[top:top+th, left:left+tw]

    return image


def random_crop(image, target, crop_size):
    """
    Random crop with security padding if the image is smaller than the crop size.
    Also check target shape, in some cases train/GT are mixed in verical/horizontal format.
    """
    h_i, w_i = image.shape[0], image.shape[1]
    h_t, w_t = target.shape[0], target.shape[1]

    # 1. Calculate necessary padding if the image is smaller than the crop
    pad_h_i = max(0, crop_size - h_i)
    pad_w_i = max(0, crop_size - w_i)
    pad_h_t = max(0, crop_size - h_t)
    pad_w_t = max(0, crop_size - w_t)

    if pad_h_i > 0 or pad_w_i > 0:
        print("padding image", pad_h_i, pad_w_i)
        # Use reflect padding to avoid black borders
        image = np.pad(image, ((0, pad_h_i), (0, pad_w_i), (0, 0)), mode="reflect")
        # Actualizamos dimensiones
        h_i, w_i = image.shape[0], image.shape[1]
    if pad_h_t > 0 or pad_w_t > 0:
        print("padding target", pad_h_t, pad_w_t)
        target = np.pad(target, ((0, pad_h_t), (0, pad_w_t), (0, 0)), mode="reflect")
        # Actualizamos dimensiones
        h_t, w_t = target.shape[0], target.shape[1]

    # 2. Crop: a SINGLE (top, left) shared by image and target so both patches
    #    stay spatially aligned.
    h = min(h_i, h_t)
    w = min(w_i, w_t)
    top = np.random.randint(0, h - crop_size + 1)
    left = np.random.randint(0, w - crop_size + 1)
    image = image[top : top + crop_size, left : left + crop_size]
    target = target[top : top + crop_size, left : left + crop_size]

    return image, target


def center_crop(image, target, crop_size):
    """Deterministic spatially aligned crop for validation."""
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)
    return (
        image[top : top + crop_size, left : left + crop_size],
        target[top : top + crop_size, left : left + crop_size],
    )


def random_flip(image, target):
    """Random horizontal and vertical flip."""
    if np.random.rand() > 0.5:
        image = np.fliplr(image)
        target = np.fliplr(target)

    if np.random.rand() > 0.5:
        image = np.flipud(image)
        target = np.flipud(target)

    return image, target


def random_rotation(image, target):
    """Random 90-degree rotation."""
    k = np.random.randint(0, 4)
    image = np.rot90(image, k=k)
    target = np.rot90(target, k=k)
    return image, target


def mod_padding_symmetric(image, factor=64):
    """Pad image so H and W are divisible by factor, using symmetric reflect padding."""
    height, width = image.shape[0], image.shape[1]

    pad_h = (factor - height % factor) % factor
    pad_w = (factor - width % factor) % factor

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    image = np.pad(
        image,
        [(pad_top, pad_bottom), (pad_left, pad_right), (0, 0)],
        mode="reflect",
    )
    return image


def set_shapes(inp, tgt, sizes):
    inp.set_shape([PATCH_SIZE, PATCH_SIZE, 3])
    tgt.set_shape([PATCH_SIZE, PATCH_SIZE, 3])
    sizes.set_shape([6])
    return inp, tgt, sizes


def read_lines_from_file(basepath, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    existing = []
    missing = []

    for line in lines:
        p = os.path.join(basepath, line)
        if os.path.exists(p):
            existing.append(p)
        else:
            missing.append(p)

    print(
        f"Checked {len(lines)} files in {basepath}: {len(existing)} existing, {len(missing)} missing."
    )
    print(f"Missing files: {missing}")

    return existing, missing


def load_image(filepath, max_retries=5, retry_delay=0.2):
    """Load an RGB image, retrying transient Google Drive I/O errors."""
    last_error = None
    for attempt in range(max_retries):
        try:
            with Image.open(filepath) as img:
                return np.asarray(img.convert("RGB"), np.float32) / 255.0
        except OSError as error:
            last_error = error
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    raise OSError(
        f"Failed to read image after {max_retries} attempts: {filepath}"
    ) from last_error


def create_dataset(data_dir, batch_size, patch_size, is_training=True):
    """Create TensorFlow dataset for training/validation."""

    print(
        f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}"
    )
    input_dir = os.path.join(data_dir, "imgs")
    target_dir = os.path.join(data_dir, "GT")
    files_list = (
        os.path.join(data_dir, "train.txt")
        if is_training
        else os.path.join(data_dir, "test.txt")
    )

    input_files, missing_inputs = read_lines_from_file(input_dir, files_list)
    target_files, missing_targets = read_lines_from_file(target_dir, files_list)
    if missing_inputs or missing_targets:
        raise FileNotFoundError(
            f"Dataset lists contain missing files in {data_dir}."
        )
    if len(input_files) != len(target_files):
        raise ValueError(f"Input/GT count mismatch in {data_dir}.")

    def load_and_preprocess(input_path, target_path):
        """Load and preprocess a single pair of images."""
        input_img = load_image(input_path.numpy().decode())
        target_img = load_image(target_path.numpy().decode())
        orig_h, orig_w = input_img.shape[:2]

        if is_training:
            # --- TRAINING FLOW ---
            # 1. Smart Random Crop (includes padding if the img is too small)
            input_img, target_img = random_crop(input_img, target_img, patch_size)

            # 2. Augmentations (Flip/Rotate)
            input_img, target_img = random_flip(input_img, target_img)
            input_img, target_img = random_rotation(input_img, target_img)

            # In training, the patch_size is already a multiple of 64.
            # No need for modd_padding_symmetric here.
            pad_h, pad_w = input_img.shape[:2]  # == patch_size
            even_h, even_w = pad_h, pad_w
        else:
            # Fixed center crop: repeated validation evaluates identical pixels.
            input_img, target_img = center_crop(input_img, target_img, patch_size)

            pad_h, pad_w = input_img.shape[:2]
            even_h, even_w = pad_h, pad_w

        # --- SAFETY CHECK / VALIDATION ---
        final_h_i, final_w_i = input_img.shape[:2]
        final_h_t, final_w_t = target_img.shape[:2]

        # Get filename for debugging (crucial to find the corrupt image)
        # We try/except because sometimes paths are not available in certain pipeline stages
        try:
            fname_i = input_path.numpy().decode('utf-8')
            fname_t = target_path.numpy().decode('utf-8')
        except:
            fname_i = "Unknown file"
            fname_t = "Unknown file"

        if is_training:
            # STRICT MODE: Training images MUST match patch_size exactly
            if final_h_i != patch_size or final_w_i != patch_size:
                error_msg = (
                    f"\n[DATASET ERROR] Invalid Training Shape!\n"
                    f"Train File: {fname_i}, Target File: {fname_t}\n"
                    f"Expected: ({patch_size}, {patch_size})\n"
                    f"Actual:   ({final_h_i}, {final_w_i})\n"
                    f"Check your random_crop logic or if training image is too small."
                )
                print(error_msg) # Print to console so you see it in logs
                raise ValueError(error_msg) # Stop training immediately
            if final_h_t != patch_size or final_w_t != patch_size:
                error_msg = (
                    f"\n[DATASET ERROR] Invalid Training Shape!\n"
                    f"Train File: {fname_i}, Target File: {fname_t}\n"
                    f"Expected: ({patch_size}, {patch_size})\n"
                    f"Actual:   ({final_h_t}, {final_w_t})\n"
                    f"Check your random_crop logic or if GT image is too small."
                )
                print(error_msg)
                raise ValueError(error_msg)


        else:
            # VALIDATION MODE: Images MUST be divisible by 64 (for MAXIM)
            if final_h_i % 64 != 0 or final_w_i % 64 != 0:
                error_msg = (
                    f"\n[DATASET ERROR] Invalid Validation Shape (Not divisible by 64)! Training image\n"
                    f"Train File: {fname_i}, Target File: {fname_t}\n"
                    f"Actual:   ({final_h_i}, {final_w_i})\n"
                    f"Remainder: ({final_h_i % 64}, {final_w_i % 64})\n"
                )
                print(error_msg)
                raise ValueError(error_msg)
            if final_h_t % 64 != 0 or final_w_t % 64 != 0:
                error_msg = (
                    f"\n[DATASET ERROR] Invalid Validation Shape (Not divisible by 64)! GT image\n"
                    f"Train File: {fname_i}, Target File: {fname_t}\n"
                    f"Actual:   ({final_h_t}, {final_w_t})\n"
                    f"Remainder: ({final_h_t % 64}, {final_w_t % 64})\n"
                )
                print(error_msg)
                raise ValueError(error_msg)

        return (
            input_img.astype(np.float32),
            target_img.astype(np.float32),
            np.array([orig_h, orig_w, even_h, even_w, pad_h, pad_w], np.int32),
        )

    dataset = tf.data.Dataset.from_tensor_slices((input_files, target_files))

    if is_training:
        dataset = dataset.shuffle(
            buffer_size=1000, seed=SEED, reshuffle_each_iteration=True
        )

    parallel_reads = min(4, os.cpu_count() or 1)
    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.float32, tf.int32],
        ),
        num_parallel_calls=parallel_reads,
        deterministic=True,
    )
    dataset = dataset.map(
        set_shapes, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True
    )

    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, len(input_files)


def resize_target_to(pred, target):
    """Downsample target if necessary to match prediction shape."""
    if pred.shape == target.shape:
        return target
    # Compute integer stride factors
    scale_h = target.shape[1] // pred.shape[1]
    scale_w = target.shape[2] // pred.shape[2]
    # Subsample target by simple stride (fast, deterministic)
    return target[:, ::scale_h, ::scale_w, :]


def compute_psnr_per_example(pred, target):
    """Return one PSNR value per image, assuming a [0, 1] range."""
    pred_255 = pred * 255.0
    target_255 = target * 255.0
    diff = pred_255 - target_255
    mse = jnp.mean(diff ** 2, axis=(1, 2, 3))
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))


def compute_psnr(pred, target):
    return jnp.mean(compute_psnr_per_example(pred, target))


def create_learning_rate_schedule(base_lr, warmup_epochs, total_steps, steps_per_epoch):
    """Create learning rate schedule with warmup and cosine decay."""
    warmup_steps = warmup_epochs * steps_per_epoch

    warmup_fn = optax.linear_schedule(
        init_value=0.0, end_value=base_lr, transition_steps=warmup_steps
    )

    cosine_fn = optax.cosine_decay_schedule(
        init_value=base_lr, decay_steps=total_steps - warmup_steps, alpha=1e-6
    )

    schedule_fn = optax.join_schedules(
        schedules=[warmup_fn, cosine_fn], boundaries=[warmup_steps]
    )

    return schedule_fn


def router_entropy_loss(probs, eps=1e-8):
    return -jnp.mean(jnp.sum(probs * jnp.log(probs + eps), axis=-1))


def load_balance_loss(probs):
    usage = jnp.mean(probs, axis=0)
    num_experts = probs.shape[-1]
    return num_experts * jnp.sum(usage**2)


def router_batch_statistics(
    probs, task_ids, sample_psnr, sample_final_loss, eps=1e-8
):
    """Return additive router statistics without collapsing expert axes.

    Every returned value is a sum or count. That makes epoch aggregation exact
    even when the last validation batch is smaller than the others.
    """
    task_ids = jnp.asarray(task_ids, dtype=jnp.int32).reshape(-1)
    sample_entropy = -jnp.sum(probs * jnp.log(probs + eps), axis=-1)
    sample_confidence = jnp.max(probs, axis=-1)
    top1 = jnp.argmax(probs, axis=-1)
    sample_psnr = jnp.asarray(sample_psnr).reshape(-1)
    sample_final_loss = jnp.asarray(sample_final_loss).reshape(-1)

    task_one_hot = jax.nn.one_hot(task_ids, NUM_TASKS, dtype=probs.dtype)
    top1_one_hot = jax.nn.one_hot(top1, NUM_EXPERTS, dtype=probs.dtype)

    return {
        "sample_count": jnp.asarray(probs.shape[0], dtype=probs.dtype),
        "router_prob_sum": jnp.sum(probs, axis=0),
        "router_top1_count": jnp.sum(top1_one_hot, axis=0),
        "router_entropy_sum": jnp.sum(sample_entropy),
        "router_confidence_sum": jnp.sum(sample_confidence),
        "router_correct_sum": jnp.sum(top1 == task_ids),
        "task_count": jnp.sum(task_one_hot, axis=0),
        "task_expert_prob_sum": task_one_hot.T @ probs,
        "task_top1_count": task_one_hot.T @ top1_one_hot,
        "task_entropy_sum": task_one_hot.T @ sample_entropy,
        "task_confidence_sum": task_one_hot.T @ sample_confidence,
        "task_psnr_sum": task_one_hot.T @ sample_psnr,
        "task_final_loss_sum": task_one_hot.T @ sample_final_loss,
    }


def summarize_router_metrics(batch_metrics):
    """Aggregate host-side batch metrics and preserve expert/task axes."""
    if not batch_metrics:
        return {}

    sample_counts = np.asarray([float(m["sample_count"]) for m in batch_metrics])
    total_samples = float(np.sum(sample_counts))

    def weighted_mean(key):
        values = np.asarray([float(m[key]) for m in batch_metrics])
        return float(np.sum(values * sample_counts) / total_samples)

    def summed(key):
        return np.sum(np.stack([np.asarray(m[key]) for m in batch_metrics]), axis=0)

    prob_sum = summed("router_prob_sum")
    top1_count = summed("router_top1_count")
    task_count = summed("task_count")
    task_prob_sum = summed("task_expert_prob_sum")
    task_top1_count = summed("task_top1_count")
    task_entropy_sum = summed("task_entropy_sum")
    task_confidence_sum = summed("task_confidence_sum")
    task_psnr_sum = summed("task_psnr_sum")
    task_final_loss_sum = summed("task_final_loss_sum")
    entropy = float(sum(float(m["router_entropy_sum"]) for m in batch_metrics) / total_samples)
    confidence = float(sum(float(m["router_confidence_sum"]) for m in batch_metrics) / total_samples)
    routing_accuracy = float(
        sum(float(m["router_correct_sum"]) for m in batch_metrics) / total_samples
    )

    task_denominator = task_count[:, None]
    task_expert_usage = np.divide(
        task_prob_sum, task_denominator,
        out=np.full_like(task_prob_sum, np.nan, dtype=np.float64),
        where=task_denominator > 0,
    )
    task_top1_frequency = np.divide(
        task_top1_count, task_denominator,
        out=np.full_like(task_top1_count, np.nan, dtype=np.float64),
        where=task_denominator > 0,
    )
    task_entropy = np.divide(
        task_entropy_sum, task_count,
        out=np.full_like(task_entropy_sum, np.nan, dtype=np.float64),
        where=task_count > 0,
    )
    task_confidence = np.divide(
        task_confidence_sum, task_count,
        out=np.full_like(task_confidence_sum, np.nan, dtype=np.float64),
        where=task_count > 0,
    )
    task_psnr = np.divide(
        task_psnr_sum, task_count,
        out=np.full_like(task_psnr_sum, np.nan, dtype=np.float64),
        where=task_count > 0,
    )
    task_final_loss = np.divide(
        task_final_loss_sum, task_count,
        out=np.full_like(task_final_loss_sum, np.nan, dtype=np.float64),
        where=task_count > 0,
    )

    return {
        "loss": weighted_mean("loss"),
        "reconstruction_loss": weighted_mean("reconstruction_loss"),
        "final_loss": weighted_mean("final_loss"),
        "auxiliary_loss": weighted_mean("auxiliary_loss"),
        "psnr": weighted_mean("psnr"),
        "balance": weighted_mean("balance"),
        "sample_count": int(total_samples),
        "router_usage": prob_sum / total_samples,
        "router_top1_frequency": top1_count / total_samples,
        "router_entropy": entropy,
        "router_normalized_entropy": entropy / np.log(NUM_EXPERTS),
        "router_effective_experts": float(np.exp(entropy)),
        "router_confidence": confidence,
        "routing_accuracy": routing_accuracy,
        "task_count": task_count.astype(np.int64),
        "task_expert_usage": task_expert_usage,
        "task_top1_frequency": task_top1_frequency,
        "task_entropy": task_entropy,
        "task_confidence": task_confidence,
        "task_psnr": task_psnr,
        "task_final_loss": task_final_loss,
    }


def print_router_diagnostics(metrics, title):
    """Print compact, named router diagnostics for one epoch."""
    if not metrics:
        return

    def expert_vector(values):
        return "  ".join(
            f"E{idx}={float(value):.3f}" for idx, value in enumerate(values)
        )

    print(f"{title}:")
    print(f"  uso soft: {expert_vector(metrics['router_usage'])}")
    print(f"  frecuencia top-1: {expert_vector(metrics['router_top1_frequency'])}")
    print(
        f"  entropia={metrics['router_entropy']:.4f} "
        f"({metrics['router_normalized_entropy']:.1%} de log(K)), "
        f"expertos efectivos={metrics['router_effective_experts']:.2f}, "
        f"confianza maxima={metrics['router_confidence']:.3f}, "
        f"accuracy tarea->experto={metrics['routing_accuracy']:.1%}"
    )
    print("  probabilidades medias tarea x experto:")
    for task_name, count, row, entropy, confidence in zip(
        TASK_NAMES, metrics["task_count"], metrics["task_expert_usage"],
        metrics["task_entropy"], metrics["task_confidence"],
    ):
        print(
            f"    {task_name:>8s} (n={int(count):5d}): {expert_vector(row)}  "
            f"H={float(entropy):.3f}  max={float(confidence):.3f}"
        )
    print("  frecuencias top-1 tarea x experto:")
    for task_name, count, row in zip(
        TASK_NAMES, metrics["task_count"], metrics["task_top1_frequency"],
    ):
        print(
            f"    {task_name:>8s} (n={int(count):5d}): {expert_vector(row)}"
        )
    print("  calidad final por tarea:")
    for task_name, count, psnr, final_loss in zip(
        TASK_NAMES, metrics["task_count"], metrics["task_psnr"],
        metrics["task_final_loss"],
    ):
        print(
            f"    {task_name:>8s} (n={int(count):5d}): "
            f"PSNR={float(psnr):7.2f} dB  final_L1={float(final_loss):.5f}"
        )


def _jsonable_metrics(metrics):
    return {
        key: value.tolist() if isinstance(value, np.ndarray) else value
        for key, value in metrics.items()
    }


def save_router_diagnostics(
    epoch, train_metrics, val_metrics, output_dir, effective_steps_per_epoch
):
    """Persist one JSON per epoch next to the model checkpoints."""
    diagnostics_dir = os.path.join(output_dir, "router_diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    path = os.path.join(diagnostics_dir, f"epoch_{epoch:03d}.json")
    payload = {
        "epoch": int(epoch),
        "routing_mode": ROUTING_MODE,
        "task_names": list(TASK_NAMES),
        "task_to_expert": {name: idx for idx, name in enumerate(TASK_NAMES)},
        "router_entropy_weight": ROUTER_ENTROPY_WEIGHT,
        "router_balance_weight": ROUTER_BALANCE_WEIGHT,
        "model_variant": MODEL_VARIANT,
        "dropout_rate": _MODEL_CONFIGS["dropout_rate"],
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "steps_per_epoch": int(effective_steps_per_epoch),
        "num_epochs": NUM_EPOCHS,
        "patch_size": PATCH_SIZE,
        "sampling_mode_train": SAMPLING_MODE,
        "load_maxim_backbone": LOAD_MAXIM_BACKBONE,
        "train": _jsonable_metrics(train_metrics),
        "validation": _jsonable_metrics(val_metrics),
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f"Router diagnostics saved to {path}")


def save_router_heatmaps(epoch, train_metrics, val_metrics, output_dir):
    """Save soft-usage and top-1 task/expert heatmaps for one epoch."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib no esta disponible; se omite el heatmap del router.")
        return

    diagnostics_dir = os.path.join(output_dir, "router_diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    panels = (
        (axes[0, 0], train_metrics["task_expert_usage"], "Train: uso soft"),
        (axes[0, 1], train_metrics["task_top1_frequency"], "Train: frecuencia top-1"),
        (axes[1, 0], val_metrics["task_expert_usage"], "Validacion: uso soft"),
        (axes[1, 1], val_metrics["task_top1_frequency"], "Validacion: frecuencia top-1"),
    )
    image = None
    for ax, matrix, title in panels:
        matrix = np.asarray(matrix, dtype=np.float64)
        image = ax.imshow(np.nan_to_num(matrix), vmin=0.0, vmax=1.0, cmap="viridis")
        ax.set_title(title)
        ax.set_xlabel("Experto")
        ax.set_ylabel("Tarea")
        ax.set_xticks(range(NUM_EXPERTS), [f"E{i}" for i in range(NUM_EXPERTS)])
        ax.set_yticks(range(NUM_TASKS), TASK_NAMES)
        for task_idx in range(NUM_TASKS):
            for expert_idx in range(NUM_EXPERTS):
                value = matrix[task_idx, expert_idx]
                label = "--" if np.isnan(value) else f"{value:.2f}"
                color = "white" if np.isnan(value) or value < 0.55 else "black"
                ax.text(expert_idx, task_idx, label, ha="center", va="center", color=color)
    fig.colorbar(image, ax=axes, shrink=0.8, label="Proporcion")
    fig.suptitle(
        f"Diagnostico de routing ({ROUTING_MODE}) - epoca {epoch}", fontsize=14
    )
    path = os.path.join(diagnostics_dir, f"epoch_{epoch:03d}_heatmaps.png")
    fig.savefig(path, dpi=160)
    plt.close(fig)
    print(f"Router heatmaps saved to {path}")


def compute_supervision_losses(
    predictions, targets, num_scales=3, auxiliary_weight=1.0
):
    """Separate MoE-final L1 from normalized deep-supervision L1.

    MAXIM orders each stage from the smallest scale to full resolution.
    The final full-resolution item is produced by the MoE; every other
    item remains an auxiliary MAXIM prediction.
    """
    if not isinstance(predictions, (list, tuple)) or not predictions:
        raise ValueError("Expected non-empty multi-stage predictions.")

    final_prediction = predictions[-1][-1]
    final_target = resize_target_to(final_prediction, targets)
    final_loss = jnp.mean(jnp.abs(final_prediction - final_target))

    auxiliary_sum = jnp.zeros_like(final_loss)
    auxiliary_weight_sum = 0.0
    last_stage = len(predictions) - 1
    for stage_idx, stage_predictions in enumerate(predictions):
        for scale_idx, prediction in enumerate(stage_predictions):
            is_moe_final = (
                stage_idx == last_stage
                and scale_idx == len(stage_predictions) - 1
            )
            if is_moe_final:
                continue
            scale_weight = 0.5 ** (num_scales - scale_idx - 1)
            target_at_scale = resize_target_to(prediction, targets)
            auxiliary_sum += scale_weight * jnp.mean(
                jnp.abs(prediction - target_at_scale)
            )
            auxiliary_weight_sum += scale_weight

    auxiliary_denominator = jnp.maximum(
        jnp.asarray(auxiliary_weight_sum, dtype=final_loss.dtype),
        jnp.asarray(1e-8, dtype=final_loss.dtype),
    )
    auxiliary_loss = auxiliary_sum / auxiliary_denominator
    reconstruction_loss = final_loss + auxiliary_weight * auxiliary_loss
    return reconstruction_loss, final_loss, auxiliary_loss, final_prediction


# Wrap at module scope so it compiles only once
def forward_apply(apply_fn, params, batch_stats, x, task_id, rngs):
    if batch_stats is not None:
        (preds, router_probs), updates = apply_fn(
            {"params": params, "batch_stats": batch_stats},
            x,
            task_id=task_id,
            train=True,
            rngs=rngs,
            mutable=["batch_stats"],
        )
        return (preds, router_probs), updates["batch_stats"]
    else:
        return apply_fn(
            {"params": params}, x, task_id=task_id, train=True, rngs=rngs
        ), None


@functools.partial(jax.jit, donate_argnums=(0,))
def train_step(state: TrainState, batch_input, batch_target, task_id, num_scales, rng):
    """Single training step (OOM-safe)."""

    def loss_fn(params, batch_stats, x, y, task_ids, rng):
        rngs = {"dropout": rng}

        # Recompute activations in backward mode instead of storing to save memory
        (preds, router_probs), new_batch_stats = jax.checkpoint(
            forward_apply, static_argnums=(0,)
        )(state.apply_fn, params, batch_stats, x, task_ids, rngs)

        reconstruction_loss, final_loss, auxiliary_loss, final_pred = (
            compute_supervision_losses(
                preds, y, num_scales, AUXILIARY_LOSS_WEIGHT
            )
        )
        entropy_loss = router_entropy_loss(router_probs)
        balance_loss = load_balance_loss(router_probs)
        final_target = resize_target_to(final_pred, y)
        sample_psnr = compute_psnr_per_example(final_pred, final_target)
        sample_final_loss = jnp.mean(
            jnp.abs(final_pred - final_target), axis=(1, 2, 3)
        )
        router_stats = router_batch_statistics(
            router_probs, task_ids, sample_psnr, sample_final_loss
        )
        psnr = jnp.mean(sample_psnr)

        total_loss = (
            reconstruction_loss
            - ROUTER_ENTROPY_WEIGHT * entropy_loss
            + ROUTER_BALANCE_WEIGHT * balance_loss
        )
        aux = {
            "reconstruction_loss": reconstruction_loss,
            "final_loss": final_loss,
            "auxiliary_loss": auxiliary_loss,
            "psnr": psnr,
            "entropy": entropy_loss,
            "balance": balance_loss,
            "router_stats": router_stats,
            "batch_stats": new_batch_stats,
        }

        return total_loss, aux

    # Don’t redeclare grad_fn every step → avoids huge graph buildup
    (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(
        state.params, state.batch_stats, batch_input, batch_target, task_id, rng
    )

    # Apply gradients safely, TrainState remains intact
    new_state = state.apply_gradients(grads=grads)

    # Update batch stats explicitly if present
    if aux["batch_stats"] is not None:
        new_state = new_state.replace(batch_stats=aux["batch_stats"])

    metrics = {
        "loss": loss,
        "reconstruction_loss": aux["reconstruction_loss"],
        "final_loss": aux["final_loss"],
        "auxiliary_loss": aux["auxiliary_loss"],
        "psnr": aux["psnr"],
        "entropy": aux["entropy"],
        "balance": aux["balance"],
        **aux["router_stats"],
    }

    return new_state, metrics


def train_epoch(state, train_dataset, num_scales, epoch, steps_per_epoch):
    """Train for one epoch."""
    batch_metrics = []
    print(f"Starting training epoch {epoch}")
    for step in range(steps_per_epoch):
        try:
            batch_input, batch_target, sizes, task_id = next(train_dataset)
        except StopIteration:
            print("Dataset exhausted early!")
            break

        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)
        task_id = jnp.asarray(task_id, dtype=jnp.int32)

        if batch_input.shape != batch_target.shape:
            print(f"Skipping step {step} due to shape mismatch")
            continue

        base_rng = jax.random.PRNGKey(epoch)
        rng = jax.random.fold_in(base_rng, step)

        state, metrics = train_step(
            state, batch_input, batch_target, task_id, num_scales, rng
        )
        metrics_np = jax.device_get(metrics)
        batch_metrics.append(metrics_np)

        if (step + 1) % LOG_EVERY == 0:
            step_usage = (
                metrics_np["router_prob_sum"] / metrics_np["sample_count"]
            )
            step_confidence = (
                metrics_np["router_confidence_sum"] / metrics_np["sample_count"]
            )
            print(
                f"Epoch {epoch}, Step {step + 1}: "
                f"loss = {metrics_np['loss']:.4f}, "
                f"final = {metrics_np['final_loss']:.4f}, "
                f"aux = {metrics_np['auxiliary_loss']:.4f}, "
                f"psnr = {metrics_np['psnr']:.2f} dB, "
                f"bal = {metrics_np['balance']:.4f}, "
                f"conf = {step_confidence:.3f}, "
                f"usage = {np.round(step_usage, 3)}"
            )

    # Compute epoch metrics
    if len(batch_metrics) > 0:
        epoch_metrics = summarize_router_metrics(batch_metrics)
        print_router_diagnostics(epoch_metrics, f"Epoch {epoch} train router")
    else:
        print("Advertencia: No se procesaron batches en esta época.")
        epoch_metrics = {}

    return state, epoch_metrics

@jax.jit
def eval_step(state, batch_input, batch_target, task_id):
    """Single evaluation step."""
    variables = {"params": state.params}
    if state.batch_stats is not None:
        variables["batch_stats"] = state.batch_stats

    predictions, router_probs = state.apply_fn(
        variables,
        batch_input,
        task_id=task_id,
        train=False,
    )

    reconstruction_loss, final_loss, auxiliary_loss, final_pred = (
        compute_supervision_losses(
            predictions, batch_target,
            _MODEL_CONFIGS["num_supervision_scales"],
            AUXILIARY_LOSS_WEIGHT,
        )
    )
    final_target = resize_target_to(final_pred, batch_target)
    sample_psnr = compute_psnr_per_example(final_pred, final_target)
    sample_final_loss = jnp.mean(
        jnp.abs(final_pred - final_target), axis=(1, 2, 3)
    )
    psnr = jnp.mean(sample_psnr)

    balance = load_balance_loss(router_probs)
    router_stats = router_batch_statistics(
        router_probs, task_id, sample_psnr, sample_final_loss
    )

    return {
        "loss": reconstruction_loss,
        "reconstruction_loss": reconstruction_loss,
        "final_loss": final_loss,
        "auxiliary_loss": auxiliary_loss,
        "psnr": psnr,
        "balance": balance,
        **router_stats,
    }


def evaluate(state, val_dataset, log_every=100, max_batches=None):
    """Evaluate on the validation set.

    Args:
      log_every: imprime PSNR/loss corrientes y throughput cada `log_every` batches.
      max_batches: si no es None, corta la evaluacion tras esa cantidad de batches
        (submuestrea la validacion para acelerar el monitoreo por epoca).
        None = pasada completa sobre el conjunto de validacion.
    """
    batch_metrics = []
    n_batches = 0
    start = time.time()
    print("Starting evaluation...")

    for batch_input, batch_target, sizes, task_id in val_dataset:
        if max_batches is not None and n_batches >= max_batches:
            break

        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)
        task_id = jnp.asarray(task_id, dtype=jnp.int32)

        metrics = jax.device_get(
            eval_step(state, batch_input, batch_target, task_id)
        )
        batch_metrics.append(metrics)
        n_batches += 1

        if n_batches % log_every == 0:
            elapsed = time.time() - start
            rate = n_batches / elapsed if elapsed > 0 else 0.0
            running = summarize_router_metrics(batch_metrics)
            print(
                f"  [eval] {n_batches} batches | "
                f"running psnr = {running['psnr']:.2f} dB, "
                f"loss = {running['loss']:.4f}, "
                f"conf = {running['router_confidence']:.3f} | {rate:.1f} batch/s"
            )

    if n_batches == 0:
        print("Evaluation: no batches processed.")
        return {}

    val_metrics = summarize_router_metrics(batch_metrics)
    elapsed = time.time() - start
    print(
        f"Evaluation done: {n_batches} batches in {elapsed:.1f}s | "
        f"psnr = {val_metrics['psnr']:.2f} dB, "
        f"loss = {val_metrics['loss']:.4f}"
    )
    print_router_diagnostics(val_metrics, "Validation router")
    return val_metrics


def create_dataset_unbatched(data_dir, patch_size, is_training=True):
    ds, length = create_dataset(
        data_dir=data_dir,
        batch_size=1,  # batch temporal
        patch_size=patch_size,
        is_training=is_training,
    )
    ds = ds.unbatch()
    return ds, length


# Add task_id (just for logging/analysis)
def add_task_id(dataset, task_id):
    task_id = tf.constant(task_id, dtype=tf.int32)
    return dataset.map(
        lambda x, y, sizes: (x, y, sizes, task_id),
        num_parallel_calls=tf.data.AUTOTUNE,
    )


def create_unified_dataset(
    task_dirs,
    batch_size,
    patch_size,
    is_training=True,
    sampling_mode="uniform",
    shuffle_buffer=1000,
):
    """
    Args:
        task_dirs: list[str] — directorios, uno por tarea
        sampling_mode (training only):
            "uniform"      -> todas las tareas equiprobables
            "proportional" -> proporcional al tamaño real
    """

    datasets = []
    lengths = []

    # Create datasets per task
    for task_id, data_dir in enumerate(task_dirs):
        ds, n = create_dataset_unbatched(
            data_dir=data_dir,
            patch_size=patch_size,
            is_training=is_training,
        )
        ds = add_task_id(ds, task_id)
        datasets.append(ds)
        lengths.append(n)

    total_samples = int(sum(lengths))
    lengths = tf.constant(lengths, dtype=tf.float32)
    # jax.debug.print("datasets lengths: {datasets} \n {lengths}", datasets, lengths)

    if is_training:
        if sampling_mode == "uniform":
            weights = tf.ones_like(lengths) / tf.cast(tf.size(lengths), tf.float32)
        elif sampling_mode == "proportional":
            weights = lengths / tf.reduce_sum(lengths)
        else:
            raise ValueError(f"Unknown sampling_mode: {sampling_mode}")
        # Each source must be infinite before mixing. Repeating only the mixed
        # dataset makes finite sources disappear when they are exhausted and
        # turns uniform sampling into dataset-size-proportional sampling.
        repeated_datasets = [dataset.repeat() for dataset in datasets]
        unified_ds = tf.data.Dataset.sample_from_datasets(
            repeated_datasets,
            weights=weights,
            seed=SEED,
            stop_on_empty_dataset=False,
        )
        unified_ds = unified_ds.shuffle(
            shuffle_buffer, seed=SEED, reshuffle_each_iteration=True
        )
    else:
        # Concatenation guarantees deterministic, exhaustive task coverage.
        unified_ds = datasets[0]
        for task_dataset in datasets[1:]:
            unified_ds = unified_ds.concatenate(task_dataset)

    unified_ds = unified_ds.batch(batch_size, drop_remainder=is_training)
    unified_ds = unified_ds.prefetch(tf.data.AUTOTUNE)

    if not is_training:
        steps_per_epoch = (total_samples + batch_size - 1) // batch_size
    elif STEPS_PER_EPOCH_OVERRIDE is not None:
        steps_per_epoch = int(STEPS_PER_EPOCH_OVERRIDE)
    else:
        steps_per_epoch = max(1, total_samples // batch_size)

    return unified_ds, steps_per_epoch

In [ ]:
def recover_tree(keys, values):
    """Recovers a tree as a nested dict from flat names and values.

    This function is useful to analyze checkpoints that are saved by our programs
    without need to access the exact source code of the experiment. In particular,
    it can be used to extract an reuse various subtrees of the scheckpoint, e.g.
    subtree of parameters.
    Args:
      keys: a list of keys, where '/' is used as separator between nodes.
      values: a list of leaf values.
    Returns:
      A nested tree-like dict.
    """
    tree = {}
    sub_trees = collections.defaultdict(list)
    for k, v in zip(keys, values):
        if "/" not in k:
            tree[k] = v
        else:
            k_left, k_right = k.split("/", 1)
            sub_trees[k_left].append((k_right, v))
    for k, kv_pairs in sub_trees.items():
        k_subtree, v_subtree = zip(*kv_pairs)
        tree[k] = recover_tree(k_subtree, v_subtree)
    return tree


def get_params(ckpt_path):
    """Get params checkpoint."""

    with tf.io.gfile.GFile(ckpt_path, "rb") as f:
        data = f.read()
    values = np.load(io.BytesIO(data))
    params = recover_tree(*zip(*values.items()))
    params = params["opt"]["target"]

    return params


def load_maxim_backbone(params_moe, params_maxim):
    moe = unfreeze(params_moe)
    flat_moe = flatten_dict(moe)
    flat_maxim = flatten_dict(params_maxim)

    for k, v in flat_maxim.items():
        if k in flat_moe and flat_moe[k].shape == v.shape:
            flat_moe[k] = v

    return freeze(unflatten_dict(flat_moe))

# Initialize Models


In [ ]:
maxim_mod = importlib.import_module("maxim.models.maxim")
router_mod = importlib.import_module("maxim.models.router")
moe_mod = importlib.import_module("maxim.models.moe")

maxim_configs = ml_collections.ConfigDict(_MODEL_CONFIGS)
maxim_configs.variant = MODEL_VARIANT
maxim_model = maxim_mod.Model(**maxim_configs)

router = router_mod.RouterModel()
# El indice de cada cabeza coincide exactamente con TASK_NAMES/task_id.
experts = [moe_mod.ExpertHead() for _ in TASK_NAMES]

moe_model = moe_mod.MaximMoE(
    maxim_model, router, experts, routing_mode=ROUTING_MODE
)

x = jnp.zeros((1, 256, 256, 3))
initial_task_id = jnp.zeros((x.shape[0],), dtype=jnp.int32)
rng = random.PRNGKey(SEED)
(initial_predictions, initial_gates), variables = moe_model.init_with_output(
    rng, x, task_id=initial_task_id, train=True
)
params_moe = variables["params"]
batch_stats = variables.get("batch_stats")

assert len(initial_predictions) == maxim_model.num_stages
assert all(
    len(stage_predictions) == _MODEL_CONFIGS["num_supervision_scales"]
    for stage_predictions in initial_predictions
)
assert initial_predictions[-1][-1].shape == x.shape
assert initial_gates.shape == (x.shape[0], NUM_EXPERTS)
if ROUTING_MODE == "oracle":
    expected_gates = jax.nn.one_hot(initial_task_id, NUM_EXPERTS)
    np.testing.assert_array_equal(np.asarray(initial_gates), np.asarray(expected_gates))
    print(f"Routing oraculo activo: {dict(enumerate(TASK_NAMES))}")
print(
    "Deep supervision activa:",
    [prediction.shape for stage in initial_predictions for prediction in stage],
)
del initial_predictions, initial_gates, initial_task_id

Routing oraculo activo: {0: 'deblur', 1: 'dehaze', 2: 'denoise', 3: 'derain', 4: 'enhance'}
Deep supervision activa: [(1, 64, 64, 3), (1, 128, 128, 3), (1, 256, 256, 3), (1, 64, 64, 3), (1, 128, 128, 3), (1, 256, 256, 3)]


# Datasets, model and state creation/restoration


In [ ]:
# @title MAXIM backbone loading (warm start)
# Carga el backbone MAXIM preentrenado (CKPT_PATH) dentro del MoE.
#
# OJO con el árbol de parámetros: en MaximMoE el backbone está anidado bajo la
# clave "maxim" (junto a "experts_0..4" y, con routing aprendido, "router"),
# así que las claves del
# checkpoint deben prefijarse con ("maxim",). La versión anterior de esta celda
# usaba load_maxim_backbone() sin prefijo: ninguna clave coincidía y NO se
# cargaba ningún parámetro (verificado: 0/1760 tensores).
# Con el prefijo, el ckpt Enhancement/LOL (S-2) cubre el backbone completo:
# 1760/1760 tensores, sin conflictos de forma.
LOAD_MAXIM_BACKBONE = False  # @param {type:"boolean"}
INIT_EXPERTS_FROM_OUTPUT_CONV = True  # @param {type:"boolean"}
VERIFY_WARM_START = True  # @param {type:"boolean"}
# Ruido gaussiano por experto al copiar la conv de salida: expertos idénticos
# con ruteo denso reciben gradientes casi idénticos y quedan clonados; el ruido
# rompe esa simetría sin alejarlos de la reconstrucción preentrenada.
EXPERT_INIT_NOISE = 1e-3

if LOAD_MAXIM_BACKBONE:
    params_maxim = get_params(CKPT_PATH)
    flat_maxim = flatten_dict(params_maxim)
    flat_moe = flatten_dict(unfreeze(params_moe))

    loaded, shape_mismatch = 0, 0
    for k, v in flat_maxim.items():
        km = ("maxim",) + k
        if km in flat_moe:
            if flat_moe[km].shape == v.shape:
                flat_moe[km] = jnp.asarray(v)
                loaded += 1
            else:
                shape_mismatch += 1
    backbone_total = sum(1 for k in flat_moe if k[0] == "maxim")
    print(f"Backbone: {loaded}/{backbone_total} tensores cargados desde el checkpoint "
          f"({shape_mismatch} descartados por forma).")
    assert loaded > 0.9 * backbone_total, (
        "Warm start incompleto: revisar CKPT_PATH y la variante del backbone."
    )

    if INIT_EXPERTS_FROM_OUTPUT_CONV:
        # La salida final de MAXIM es conv(features) + x, igual que la mezcla
        # residual del MoE (x + sum_k p_k E_k(features)), así que copiar la conv
        # de salida del último stage hace que el MoE arranque reproduciendo la
        # reconstrucción del modelo preentrenado.
        out_keys = [k for k in params_maxim
                    if k.startswith("stage_") and k.endswith("_output_conv_0")]
        final_key = max(out_keys, key=lambda k: int(k.split("_")[1]))
        kern = jnp.asarray(params_maxim[final_key]["kernel"])
        bias = jnp.asarray(params_maxim[final_key]["bias"])
        noise_rng = random.PRNGKey(SEED)
        for e in range(NUM_EXPERTS):
            noise_rng, sub = random.split(noise_rng)
            noise = EXPERT_INIT_NOISE * random.normal(sub, kern.shape, kern.dtype)
            flat_moe[(f"experts_{e}", "output_conv", "kernel")] = kern + noise
            flat_moe[(f"experts_{e}", "output_conv", "bias")] = bias
        print(f"Expertos: {NUM_EXPERTS} cabezas inicializadas desde '{final_key}' "
              f"(+ ruido sigma={EXPERT_INIT_NOISE:g} por experto).")

    params_moe = freeze(unflatten_dict(flat_moe))

    if VERIFY_WARM_START:
        # Con expertos ~idénticos, la salida del MoE debe reproducir la del
        # MAXIM preentrenado sobre la misma entrada (diff ~ EXPERT_INIT_NOISE),
        # sea cual sea la distribución del ruteador.
        x_test = random.uniform(random.PRNGKey(0), (1, 256, 256, 3))
        y_maxim = maxim_model.apply({"params": params_maxim}, x_test, train=False)
        y_maxim = y_maxim[-1][-1]  # última etapa, resolución completa
        test_task_id = jnp.zeros((x_test.shape[0],), dtype=jnp.int32)
        variables_test = {"params": params_moe}
        if batch_stats is not None:
            variables_test["batch_stats"] = batch_stats
        y_moe, gates = moe_model.apply(
            variables_test, x_test, task_id=test_task_id, train=False
        )
        y_moe_final = y_moe[-1][-1]
        diff = jnp.mean(jnp.abs(y_maxim - y_moe_final))
        print(f"Verificación: |MoE - MAXIM preentrenado| = {diff:.2e} "
              f"(esperado ~{EXPERT_INIT_NOISE:g})")
        print(f"Gates medios ({ROUTING_MODE=}): {jnp.mean(gates, axis=0)}")
else:
    print("LOAD_MAXIM_BACKBONE=False: backbone con inicialización aleatoria.")

LOAD_MAXIM_BACKBONE=False: backbone con inicialización aleatoria.


In [ ]:
# @title Build datasets, model and state
try:
    train_dataset, steps_per_epoch = create_unified_dataset(
        task_dirs=TASK_DIRS,
        batch_size=BATCH_SIZE,
        patch_size=PATCH_SIZE,
        is_training=True,
        sampling_mode=SAMPLING_MODE,
    )


    test_dataset, val_steps = create_unified_dataset(
        task_dirs=TASK_DIRS,
        batch_size=EVAL_BATCH_SIZE,
        patch_size=EVAL_PATCH_SIZE,
        is_training=False,
    )
    print(f"Training steps per epoch: {steps_per_epoch}")
    print(f"Full validation batches per epoch: {val_steps}")
    if STEPS_PER_EPOCH_OVERRIDE is not None:
        assert steps_per_epoch == STEPS_PER_EPOCH_OVERRIDE

    # Calculate steps
    total_steps = steps_per_epoch * NUM_EPOCHS

    # 2. Setup Optimizer and State
    learning_rate_fn = create_learning_rate_schedule(
        LEARNING_RATE, WARMUP_EPOCHS, total_steps, steps_per_epoch
    )

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY),
    )

    # Create TrainState
    state = TrainState.create(
        apply_fn=moe_model.apply, params=params_moe, tx=tx, batch_stats=batch_stats
    )

    if RUN_PREFLIGHT:
        preflight_input, preflight_target, _, preflight_task_id = next(
            iter(test_dataset)
        )
        preflight_metrics = jax.device_get(
            eval_step(
                state, jnp.asarray(preflight_input), jnp.asarray(preflight_target),
                jnp.asarray(preflight_task_id, dtype=jnp.int32),
            )
        )
        for metric_name in ("loss", "final_loss", "auxiliary_loss", "psnr"):
            if not np.isfinite(float(preflight_metrics[metric_name])):
                raise FloatingPointError(
                    f"Preflight produced non-finite {metric_name}."
                )
        if float(preflight_metrics["router_correct_sum"]) != float(
            preflight_metrics["sample_count"]
        ):
            raise RuntimeError("Oracle routing failed during preflight.")
        print(
            f"Preflight OK | task_id={np.asarray(preflight_task_id).tolist()} | "
            f"loss={float(preflight_metrics['loss']):.4f} | "
            f"psnr={float(preflight_metrics['psnr']):.2f} dB"
        )
except Exception as e:
    print(f"Training setup failed: {e}")
    import traceback
    traceback.print_exc()

Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/imgs: 8680 existing, 0 missing.
Missing files: []
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/GT: 8680 existing, 0 missing.
Missing files: []
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/imgs: 909 existing, 0 missing.
Missing files: []
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/GT: 909 existing, 0 missing.
Missing files: []
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise
Checked 4450 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/imgs: 4450 existing, 0 missing.
Missing files: []
Checked 4450 files in /co

In [ ]:
# @title Load Checkpoints / Resume
# Reanudar una corrida interrumpida o empezar limpio.
#   "none"     -> no cargar nada (usa el backbone warm-started / init aleatoria
#                 de la celda anterior).
#   "latest"   -> reanuda desde el checkpoint mas reciente en OUTPUT_DIR
#                 (RECOMENDADO para retomar una corrida interrumpida).
#   "best"     -> carga best_checkpoint (mejor PSNR; util para evaluar, no ideal
#                 para reanudar porque puede perder epocas posteriores al mejor).
#   "specific" -> carga OUTPUT_DIR/checkpoint_{epoch_to_load}.
#
# NOTA: los checkpoints previos a 2026-07-05 (random_crop desalineado) no deben
# reutilizarse; borrar o renombrar OUTPUT_DIR antes de una corrida nueva.
RESUME_MODE = "latest"  # @param ["none", "latest", "best", "specific"]
epoch_to_load = None      # @param {type:"integer"}

start_epoch = 0
best_psnr = -np.inf


def _find_latest_epoch(output_dir):
    """Mayor N entre los checkpoints OUTPUT_DIR/checkpoint_<N> (ignora best_checkpoint)."""
    if not os.path.isdir(output_dir):
        return None
    eps = [int(n.split("_")[-1]) for n in os.listdir(output_dir)
           if n.startswith("checkpoint_") and n.split("_")[-1].isdigit()]
    return max(eps) if eps else None


if RESUME_MODE == "none":
    existing_epoch = _find_latest_epoch(OUTPUT_DIR)
    if existing_epoch is not None and not ALLOW_OVERWRITE_EXISTING_RUN:
        raise FileExistsError(
            f"OUTPUT_DIR ya contiene checkpoint_{existing_epoch}. Usa "
            "RESUME_MODE='latest', cambia OUTPUT_DIR o habilita explícitamente "
            "ALLOW_OVERWRITE_EXISTING_RUN."
        )
    print("RESUME_MODE=none: corrida nueva desde inicialización aleatoria.")
else:
    if RESUME_MODE == "latest":
        ckpt_dir = OUTPUT_DIR
        if _find_latest_epoch(OUTPUT_DIR) is None:
            raise FileNotFoundError(
                f"RESUME_MODE='latest' pero no hay checkpoints en {OUTPUT_DIR}."
            )
    elif RESUME_MODE == "best":
        ckpt_dir = os.path.join(OUTPUT_DIR, "best_checkpoint")
    elif RESUME_MODE == "specific":
        ckpt_dir = os.path.join(OUTPUT_DIR, f"checkpoint_{epoch_to_load}")
        if not os.path.exists(ckpt_dir):
            raise FileNotFoundError(f"No existe {ckpt_dir}.")
    else:
        raise ValueError(f"RESUME_MODE invalido: {RESUME_MODE}")

    # restore_checkpoint NO lanza excepcion si no encuentra nada: devuelve el
    # target intacto. Detectamos ese caso comparando el contador de pasos.
    step_before = int(state.step)
    state = checkpoints.restore_checkpoint(ckpt_dir=ckpt_dir, target=state)
    if int(state.step) == step_before:
        raise FileNotFoundError(
            f"restore_checkpoint no cargo nada desde {ckpt_dir} "
            f"(state.step sigue en {step_before}). Revisar RESUME_MODE / la ruta / epoch_to_load."
        )

    # start_epoch se infiere del contador REAL del optimizador (no del nombre del
    # archivo), asi el bucle y el LR schedule quedan alineados con el progreso real.
    start_epoch = round(int(state.step) / steps_per_epoch)

    # Recuperar el mejor PSNR historico para no pisar best_checkpoint con algo peor.
    best_metric_path = os.path.join(OUTPUT_DIR, "best_checkpoint", "best_metric.json")
    if os.path.exists(best_metric_path):
        with open(best_metric_path) as f:
            best_psnr = float(json.load(f).get("psnr", -np.inf))
        print(f"best_psnr historico recuperado: {best_psnr:.2f} dB")
    else:
        print("Aviso: sin best_metric.json; best_psnr arranca en -inf.")

    print(f"Resume OK desde {ckpt_dir} | state.step={int(state.step)} -> "
          f"epocas completadas={start_epoch}, continua en la epoca {start_epoch + 1} "
          f"(hasta NUM_EPOCHS={NUM_EPOCHS}).")
    if start_epoch >= NUM_EPOCHS:
        print(f"  ADVERTENCIA: start_epoch ({start_epoch}) >= NUM_EPOCHS ({NUM_EPOCHS}); "
              f"el bucle no correra ninguna epoca. Subir NUM_EPOCHS para seguir entrenando.")

best_psnr historico recuperado: 25.55 dB
Resume OK desde /content/gdrive/MyDrive/Facultad/tesis/ckpts/moe_oracle_all_S-2_scratch | state.step=14000 -> epocas completadas=7, continua en la epoca 8 (hasta NUM_EPOCHS=30).


# Main Training Loop


In [ ]:
# @title Main Training Loop
# NUM_EPOCHS es el TOTAL de epocas (no adicionales): al reanudar se corren solo
# las restantes, y el LR schedule (dimensionado a NUM_EPOCHS) queda alineado.
class TeeStream:
    """Mirror stdout/stderr to the Colab cell and a persistent file."""

    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


os.makedirs(OUTPUT_DIR, exist_ok=True)
_console_stdout = sys.stdout
_console_stderr = sys.stderr
_training_log_file = open(
    TRAIN_LOG_PATH, "a", encoding="utf-8", buffering=1
)
sys.stdout = TeeStream(_console_stdout, _training_log_file)
sys.stderr = TeeStream(_console_stderr, _training_log_file)
print("\n" + "=" * 80)
print(
    f"Training session started: {time.strftime('%Y-%m-%d %H:%M:%S')} | "
    f"resume={RESUME_MODE} | start_epoch={start_epoch} | state.step={int(state.step)}"
)
print(f"Persistent log: {TRAIN_LOG_PATH}")

try:
    start_epoch = globals().get("start_epoch", 0)
    best_psnr = globals().get("best_psnr", -np.inf)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    best_ckpt_path = os.path.join(OUTPUT_DIR, "best_checkpoint")

    print(f"Starting training: epocas {start_epoch + 1}..{NUM_EPOCHS} "
          f"(best_psnr previo: {best_psnr:.2f} dB)")
    train_dataset_iterator = iter(train_dataset)

    for epoch in range(start_epoch, NUM_EPOCHS):
        real_epoch = epoch + 1

        state, train_metrics = train_epoch(
            state, train_dataset_iterator,
            _MODEL_CONFIGS["num_supervision_scales"], real_epoch, steps_per_epoch,
        )
        print(
            f"Epoch {real_epoch} train: loss={train_metrics['loss']:.4f}, "
            f"final={train_metrics['final_loss']:.4f}, "
            f"aux={train_metrics['auxiliary_loss']:.4f}, "
            f"psnr={train_metrics['psnr']:.2f} dB, "
            f"balance={train_metrics['balance']:.4f}"
        )

        val_metrics = evaluate(state, test_dataset, log_every=EVAL_LOG_EVERY, max_batches=MAX_EVAL_BATCHES)
        if MAX_EVAL_BATCHES is None:
            if np.any(np.asarray(val_metrics["task_count"]) == 0):
                raise RuntimeError(
                    f"Validation omitted tasks: counts={val_metrics['task_count']}"
                )
        print(
            f"Epoch {real_epoch} validation: loss={val_metrics['loss']:.4f}, "
            f"final={val_metrics['final_loss']:.4f}, "
            f"aux={val_metrics['auxiliary_loss']:.4f}, "
            f"psnr={val_metrics['psnr']:.2f} dB, "
            f"balance={val_metrics['balance']:.4f}"
        )
        save_router_diagnostics(
            real_epoch, train_metrics, val_metrics, OUTPUT_DIR, steps_per_epoch
        )
        save_router_heatmaps(
            real_epoch, train_metrics, val_metrics, OUTPUT_DIR
        )
        cur_psnr = val_metrics["psnr"]

        is_best = cur_psnr > best_psnr
        # Guardado periodico (para reanudar) y del mejor modelo.
        if (real_epoch % SAVE_EVERY == 0) or is_best:
            try:
                # FLAT en OUTPUT_DIR con keep=5: flax poda los viejos y 'latest'
                # resuelve solo por numero de paso. (Subir keep para conservar mas.)
                checkpoints.save_checkpoint(
                    ckpt_dir=OUTPUT_DIR, target=state, step=real_epoch,
                    keep=5, overwrite=True,
                )
                print(f"Saved checkpoint (epoch {real_epoch}) in {OUTPUT_DIR}")
            except Exception as e:
                print(f"Failed to save checkpoint: {e}")

            if is_best:
                best_psnr = cur_psnr
                try:
                    checkpoints.save_checkpoint(
                        ckpt_dir=best_ckpt_path, target=state, step=real_epoch,
                        keep=1, overwrite=True,
                    )
                    # Persistir el mejor PSNR para sobrevivir a un resume.
                    with open(os.path.join(best_ckpt_path, "best_metric.json"), "w") as f:
                        json.dump({"psnr": float(best_psnr), "epoch": int(real_epoch)}, f)
                    print(f"New best model! PSNR: {best_psnr:.2f} dB (epoch {real_epoch})")
                except Exception as e:
                    print(f"Failed to save best checkpoint: {e}")

    print("Training finished.")
except Exception as e:
    print(f"Training loop failed: {e}")
    import traceback
    traceback.print_exc()
finally:
    print(f"Training session ended: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Complete console log saved to {TRAIN_LOG_PATH}")
    sys.stdout.flush()
    sys.stderr.flush()
    sys.stdout = _console_stdout
    sys.stderr = _console_stderr
    _training_log_file.close()

Streaming output truncated to the last 5000 lines.
Epoch 14, Step 390: loss = 0.0149, final = 0.0054, aux = 0.0095, psnr = 43.15 dB, bal = 2.5000, conf = 1.000, usage = [0.  0.5 0.5 0.  0. ]
Epoch 14, Step 400: loss = 0.1275, final = 0.0352, aux = 0.0923, psnr = 27.41 dB, bal = 5.0000, conf = 1.000, usage = [0. 0. 0. 0. 1.]
Epoch 14, Step 410: loss = 0.3039, final = 0.1398, aux = 0.1641, psnr = 17.04 dB, bal = 2.5000, conf = 1.000, usage = [0.5 0.  0.  0.  0.5]
Epoch 14, Step 420: loss = 0.1508, final = 0.0723, aux = 0.0785, psnr = 19.99 dB, bal = 2.5000, conf = 1.000, usage = [0.  0.5 0.  0.5 0. ]
Epoch 14, Step 430: loss = 0.0194, final = 0.0088, aux = 0.0106, psnr = 38.55 dB, bal = 5.0000, conf = 1.000, usage = [0. 0. 1. 0. 0.]
Epoch 14, Step 440: loss = 0.0805, final = 0.0380, aux = 0.0425, psnr = 25.60 dB, bal = 2.5000, conf = 1.000, usage = [0.  0.5 0.  0.5 0. ]
Epoch 14, Step 450: loss = 0.0672, final = 0.0311, aux = 0.0362, psnr = 28.26 dB, bal = 2.5000, conf = 1.000, usage = [

# Validation and Visualizer

In [ ]:
# test_dataset, val_steps = create_unified_dataset(
#         task_dirs=TASK_DIRS,
#         batch_size=BATCH_SIZE,
#         patch_size=PATCH_SIZE,
#         is_training=False,
#         sampling_mode="proportional",
#     )
val_metrics = evaluate(state, test_dataset, log_every=EVAL_LOG_EVERY, max_batches=MAX_EVAL_BATCHES)
print(f"Epoch {epoch} Validation Metrics: {val_metrics}")

Starting evaluation...
  [eval] 100 batches | running psnr = 23.56 dB, loss = 0.0905, conf = 1.000 | 3.9 batch/s
  [eval] 200 batches | running psnr = 25.74 dB, loss = 0.0727, conf = 1.000 | 4.1 batch/s
  [eval] 300 batches | running psnr = 25.12 dB, loss = 0.0723, conf = 1.000 | 4.2 batch/s
  [eval] 400 batches | running psnr = 25.03 dB, loss = 0.0746, conf = 1.000 | 4.2 batch/s
  [eval] 500 batches | running psnr = 25.82 dB, loss = 0.0691, conf = 1.000 | 4.2 batch/s
  [eval] 600 batches | running psnr = 26.19 dB, loss = 0.0661, conf = 1.000 | 4.2 batch/s
  [eval] 700 batches | running psnr = 25.58 dB, loss = 0.0734, conf = 1.000 | 4.3 batch/s
  [eval] 800 batches | running psnr = 25.37 dB, loss = 0.0753, conf = 1.000 | 4.3 batch/s
  [eval] 900 batches | running psnr = 25.53 dB, loss = 0.0735, conf = 1.000 | 4.3 batch/s
  [eval] 1000 batches | running psnr = 25.39 dB, loss = 0.0747, conf = 1.000 | 4.2 batch/s
  [eval] 1100 batches | running psnr = 25.73 dB, loss = 0.0727, conf = 1.000

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import requests
from PIL import Image
from io import BytesIO


def resize(path, size):
    """Load and resize image preserving aspect ratio."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = size / max(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return img.resize((new_w, new_h), Image.LANCZOS)

def pre_process(pil_img):
    """Prepare image for MAXIM (divisible by 64)."""
    w, h = pil_img.size
    # Resize to be multiple of 64 for MAXIM architecture requirements
    new_w = (w // 64) * 64
    new_h = (h // 64) * 64
    if new_w != w or new_h != h:
        pil_img = pil_img.resize((new_w, new_h), Image.LANCZOS)

    img = np.asarray(pil_img, dtype=np.float32) / 255.0
    img = np.expand_dims(img, axis=0) # Add batch dim
    return img, h, w, new_h, new_w

def make_shape_even(image):
  """Pad the image to have even shapes."""
  height, width = image.shape[0], image.shape[1]
  padh = 1 if height % 2 != 0 else 0
  padw = 1 if width % 2 != 0 else 0
  image = jnp.pad(image, [(0, padh), (0, padw), (0, 0)], mode='reflect')
  return image

def load_image(filepath):
    """Load and preprocess image."""
    img = Image.open(filepath).convert("RGB")
    img = np.asarray(img, np.float32) / 255.0
    return img

def pre_process_bytes(input_file):
  '''
  Pre-process the image before sending to the model
  '''
  input_img = load_image(input_file)
  # Padding images to have even shapes
  height, width = input_img.shape[0], input_img.shape[1]
  input_img = make_shape_even(input_img)
  height_even, width_even = input_img.shape[0], input_img.shape[1]

  # padding images to be multiplies of 64
  input_img = mod_padding_symmetric(input_img, factor=64)
  input_img = np.expand_dims(input_img, axis=0)

  return input_img, height, width, height_even, width_even

def predict(input_img, task_name):
    """Run inference using the global 'state'."""
    if task_name not in TASK_NAMES:
        raise ValueError(f"Unknown task {task_name!r}; expected one of {TASK_NAMES}.")
    task_id = jnp.full(
        (input_img.shape[0],), TASK_NAMES.index(task_name), dtype=jnp.int32
    )
    variables = {'params': state.params}
    if state.batch_stats is not None:
        variables['batch_stats'] = state.batch_stats

    # Forward pass (train=False)
    # The model returns (prediction, gates)
    predictions, gates = state.apply_fn(
        variables, input_img, task_id=task_id, train=False
    )
    return predictions[-1][-1]

def post_process(pred, h_orig, w_orig, h_even, w_even):
    """Convert prediction back to PIL image."""
    pred = np.array(pred[0]) # Remove batch dim
    pred = np.clip(pred, 0.0, 1.0)
    pred = (pred * 255).astype(np.uint8)
    return Image.fromarray(pred)

# --- Main Execution ---
# path = "/content/gdrive/MyDrive/Facultad/tesis/test_imgs/IMG_6425.JPG"
path = False
PREDICT_TASK = "enhance"

# url = "https://phototraces.b-cdn.net/wp-content/uploads/2021/02/id_Free_RAW_Photos_for_Editing_09_Uneditedd.jpg"
url = "https://phototraces.b-cdn.net/wp-content/uploads/2021/03/Free_RAW_Photos_for_Editing_13_Unedited.jpg"

if path and os.path.exists(path):
    # 1. Load and Resize (Using 1280 to fit in Colab memory easily, adjust if needed)
    input_pil_img = resize(path, 1280)

    # 2. Preprocess
    input_img, h, w, h_even, w_even = pre_process(input_pil_img)

    # 3. Inference
    print("Running inference...")
    preds = predict(input_img, PREDICT_TASK)

    # 4. Post-process
    result = post_process(preds, h, w, h_even, w_even)

    # 5. Visualize
    f, ax = plt.subplots(1, 2, figsize=(25, 15))

    ax[0].imshow(input_pil_img)
    ax[0].set_title("Original Image (Resized Input)")
    ax[0].axis('off')

    ax[1].imshow(result)
    ax[1].set_title("Enhanced Image")
    ax[1].axis('off')

    plt.show()
elif url:
  image_bytes = BytesIO(requests.get(url).content)
  input_img, height, width, height_even, width_even = pre_process_bytes(image_bytes)
  preds = predict(input_img, PREDICT_TASK)
  result = post_process(preds, height, width, height_even, width_even)
  f, ax = plt.subplots(1, 2, figsize=(25, 15))

  ax[0].imshow(np.array(Image.open(image_bytes)))
  ax[0].set_title("Original Image (Resized Input)")
  ax[0].axis('off')

  ax[1].imshow(result)
  ax[1].set_title("Enhanced Image")
  ax[1].axis('off')

  plt.show()
  print(compute_psnr(preds[0], input_img))

else:
    print(f"Image not found: {path}, {url}")

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7e573144a9d0>